# Fine-tuning the two offensive-language classifiers

Fine-tunes **KcBERT** (`beomi/kcbert-base`) and **KLUE-BERT** (`klue/bert-base`) for binary offensiveness (KOLD level A) on the same split, then freezes them. All later analyses use these frozen classifiers.

- Input: comment text only (no title), max 128 tokens
- 1,292 explicit-target comments (`data/splits/excluded_eval_pool.json`) are held out from fine-tuning
- Stratified 80/10/10 split of the rest with seed 42; AdamW, lr 2e-5, batch 32, 3 epochs, 10% linear warmup, fp16 autocast; the epoch with the best validation F1 is kept
- Runs on a Colab T4 GPU in about 40 minutes for both models

Retraining with the same split and seed gives test F1 of about 0.83 (KcBERT) and 0.82 (KLUE-BERT); GPU nondeterminism can change the last digits. Place the resulting `f_kcbert/` and `f_klue/` folders under `checkpoints/`.


In [ ]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, json, random, numpy as np, torch, urllib.request
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

assert torch.cuda.is_available(), "Select a GPU runtime (e.g. T4)."
print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MAX_LEN   = 128          # longest KOLD comment is about 110 tokens
BATCH     = 32
EPOCHS    = 3
LR        = 2e-5
MODELS    = {"kcbert": "beomi/kcbert-base", "klue": "klue/bert-base"}


In [ ]:
# KOLD (Jeong et al., 2022) from the official repository
KOLD_URL = "https://media.githubusercontent.com/media/boychaboy/KOLD/main/data/kold_v1.json"
urllib.request.urlretrieve(KOLD_URL, "kold_v1.json")
kold = json.load(open("kold_v1.json", encoding="utf-8"))
assert len(kold) == 40429
print("KOLD rows:", len(kold))

# held-out explicit-target comments (from this repository)
REPO_RAW = "https://raw.githubusercontent.com/se-eonkim/ko-repr-intervention-validity/main"
urllib.request.urlretrieve(f"{REPO_RAW}/data/splits/excluded_eval_pool.json", "excluded_eval_pool.json")

In [ ]:
EXCLUDE_GUIDS = set(json.load(open("excluded_eval_pool.json", encoding="utf-8"))["guids"])
assert len(EXCLUDE_GUIDS) == 1292

# input = comment only; label = OFF (level-A offensiveness)
rows = [(r["guid"], r["comment"], int(bool(r["OFF"])))
        for r in kold
        if r["guid"] not in EXCLUDE_GUIDS and r.get("comment", "").strip()]
guid_all = [g for g, _, _ in rows]
X = [c for _, c, _ in rows]
y = [lab for _, _, lab in rows]
print(f"training pool: {len(rows)} | offensive {sum(y)} / not {len(y) - sum(y)}")

In [ ]:
# stratified 80/10/10 split -> p2_split.json
idx = list(range(len(rows)))
tr, tmp = train_test_split(idx, test_size=0.20, stratify=y, random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.50, stratify=[y[i] for i in tmp], random_state=SEED)

def pick(ids): return [X[i] for i in ids], [y[i] for i in ids]
Xtr, ytr = pick(tr); Xva, yva = pick(va); Xte, yte = pick(te)
print(f"train {len(tr)} / val {len(va)} / test {len(te)}")

split_meta = {"seed": SEED, "max_len": MAX_LEN,
              "train_guids": [guid_all[i] for i in tr],
              "val_guids":   [guid_all[i] for i in va],
              "test_guids":  [guid_all[i] for i in te]}
json.dump(split_meta, open("p2_split.json", "w", encoding="utf-8"), ensure_ascii=False)


In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
device = "cuda"

class DS(Dataset):
    def __init__(self, texts, labels): self.t, self.y = texts, labels
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.t[i], self.y[i]

def make_collate(tok):
    def collate(batch):
        texts, labels = zip(*batch)
        enc = tok(list(texts), truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(labels)
        return enc
    return collate

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); preds, gts = [], []
    for batch in loader:
        labels = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**batch).logits
        preds += logits.argmax(-1).cpu().tolist(); gts += labels.tolist()
    # binary F1 of the offensive class (used to pick the epoch) and macro F1
    return f1_score(gts, preds), accuracy_score(gts, preds), f1_score(gts, preds, average="macro")

def train_one(name, hf_id):
    print(f"\n===== {name} ({hf_id}) =====")
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=2).to(device)
    collate = make_collate(tok)
    tr = DataLoader(DS(Xtr, ytr), batch_size=BATCH, shuffle=True,  collate_fn=collate)
    va = DataLoader(DS(Xva, yva), batch_size=64,    shuffle=False, collate_fn=collate)
    te = DataLoader(DS(Xte, yte), batch_size=64,    shuffle=False, collate_fn=collate)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(tr) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda")

    best_f1, best_state = -1.0, None
    for ep in range(EPOCHS):
        model.train()
        for step, batch in enumerate(tr):
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            opt.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch, labels=labels).loss
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            if step % 100 == 0:
                print(f"  ep{ep} step {step}/{len(tr)} loss {loss.item():.4f}")
        vf1, vacc, _ = evaluate(model, va)
        print(f"  [ep{ep}] val F1 {vf1:.4f} acc {vacc:.4f}")
        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    tf1, tacc, tmacro = evaluate(model, te)
    print(f"  >>> {name} TEST  F1 {tf1:.4f}  macro-F1 {tmacro:.4f}  acc {tacc:.4f}")
    save_dir = f"f_{name}"                         # saved in fp16
    model.half().save_pretrained(save_dir); tok.save_pretrained(save_dir)
    return {"val_f1": round(best_f1, 4), "test_f1": round(tf1, 4),
            "test_macro_f1": round(tmacro, 4), "test_acc": round(tacc, 4)}


In [ ]:
all_metrics = {}
for name, hf_id in MODELS.items():
    all_metrics[name] = train_one(name, hf_id)
json.dump(all_metrics, open("p2_metrics.json","w"), ensure_ascii=False, indent=2)
print("\n=== summary ===\n", json.dumps(all_metrics, indent=2))


In [ ]:
# download the two checkpoints, the split and the metrics
import zipfile
with zipfile.ZipFile("p2_frozen_f.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for d in ("f_kcbert", "f_klue"):
        for root,_,files in os.walk(d):
            for fn in files:
                fp = os.path.join(root, fn); z.write(fp, fp)
    z.write("p2_split.json"); z.write("p2_metrics.json")
print("zip size (MB):", round(os.path.getsize("p2_frozen_f.zip")/1e6, 1))

from google.colab import files
files.download("p2_frozen_f.zip")
